In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pareto, norm, cauchy, expon, gamma as gamma_dist
import warnings
warnings.filterwarnings('ignore')



def quantile_fn(sample, q):
    if len(sample) == 0:
        return np.nan
    return np.quantile(sample, q)

def hill_estimator_tail_index(x, k):
    if len(x) <= k:
        k = len(x) - 1
    if k <= 1:
        return np.nan

    x_sorted = np.sort(x)[::-1]

    xk1 = x_sorted[k]
    if xk1 <= 0:
        return np.nan

    logs = np.log(x_sorted[:k] / xk1)
    mean_log = np.mean(logs)

    if mean_log <= 0:
        return np.nan
    return 1.0 / mean_log

def regularized_hill_alpha(x, k_min=5, alpha_0=2.0, weight_prior=10.0):
    n = len(x)
    if n < k_min:
        return alpha_0


    k = int(n**0.65)
    k = np.clip(k, k_min, n // 2)

    hill = hill_estimator_tail_index(x, k)

    if np.isnan(hill) or hill < 0.1:
        return alpha_0

    alpha = (k * hill + weight_prior * alpha_0) / (k + weight_prior)
    return np.clip(alpha, 1.01, 10.0)

class FunctionalUCB:
    def __init__(self, K, q, c=2.0, alpha_0=2.0, weight_prior=10.0, k_min=5):
        self.K = K
        self.q = q
        self.c = c
        self.alpha_0 = alpha_0
        self.weight_prior = weight_prior
        self.k_min = k_min
        self.samples = [[] for _ in range(K)]
        self.t = 0

    def reset(self):
        self.samples = [[] for _ in range(self.K)]
        self.t = 0

    def select_arm(self):
        self.t += 1

        if self.t <= self.K:
            return self.t - 1

        ucb_values = np.zeros(self.K)

        for i in range(self.K):
            smp = self.samples[i]
            n_i = len(smp)


            if n_i == 0:
                ucb_values[i] = np.inf
                continue

            qv = quantile_fn(smp, self.q)
            if np.isnan(qv):
                ucb_values[i] = np.inf
                continue


            if n_i < self.k_min:
                alpha = self.alpha_0
            else:
                alpha = regularized_hill_alpha(
                    smp,
                    k_min=self.k_min,
                    alpha_0=self.alpha_0,
                    weight_prior=self.weight_prior
                )

            bonus = self.c * (np.log(self.t) / n_i) ** (1.0 / alpha)
            ucb_values[i] = qv + bonus

        return np.argmax(ucb_values)

    def update(self, arm, reward):
        self.samples[arm].append(reward)


class QuantileUCB:
    def __init__(self, K, q=0.75, c=1.0):
        self.K = K
        self.q = q
        self.c = c
        self.samples = [[] for _ in range(K)]
        self.t = 0

    def reset(self):
        self.samples = [[] for _ in range(self.K)]
        self.t = 0

    def select_arm(self):
        self.t += 1
        if self.t <= self.K:
            return self.t - 1
        ucb = np.full(self.K, -np.inf)
        for i in range(self.K):
            smp = self.samples[i]
            if len(smp) == 0:
                continue
            qv = quantile_fn(smp, self.q)
            if np.isnan(qv):
                continue
            bonus = self.c * np.sqrt(np.log(self.t) / len(smp))
            ucb[i] = qv + bonus
        return np.argmax(ucb)

    def update(self, arm, reward):
        self.samples[arm].append(reward)

class ClassicUCBForQuantile:
    def __init__(self, K, q=0.75, c=1.0):
        self.K = K
        self.q = q
        self.c = c
        self.samples = [[] for _ in range(K)]
        self.t = 0

    def reset(self):
        self.samples = [[] for _ in range(self.K)]
        self.t = 0

    def select_arm(self):
        self.t += 1
        if self.t <= self.K:
            return self.t - 1

        ucb = np.full(self.K, -np.inf)
        for i in range(self.K):
            smp = self.samples[i]
            if len(smp) == 0:
                continue

            qv = np.quantile(smp, self.q)
            bonus = self.c * np.sqrt(np.log(self.t) / len(smp))
            ucb[i] = qv + bonus

        return np.argmax(ucb)

    def update(self, arm, reward):
        self.samples[arm].append(reward)


class HeavyTailUCBForQuantile:

    def __init__(self, K, q=0.75, alpha=1.5, c=1.0):
        self.K = K
        self.q = q
        self.alpha = alpha
        self.c = c
        self.samples = [[] for _ in range(K)]
        self.t = 0

    def reset(self):
        self.samples = [[] for _ in range(self.K)]
        self.t = 0

    def select_arm(self):
        self.t += 1
        if self.t <= self.K:
            return self.t - 1

        ucb = np.full(self.K, -np.inf)
        for i in range(self.K):
            smp = self.samples[i]
            if len(smp) == 0:
                continue

            qv = np.quantile(smp, self.q)

            bonus = self.c * (np.log(self.t) / len(smp)) ** (1.0 / self.alpha)
            ucb[i] = qv + bonus

        return np.argmax(ucb)

    def update(self, arm, reward):
        self.samples[arm].append(reward)


class AdaptiveHeavyTailUCBForQuantile:

    def __init__(self, K, q=0.75, c=1.0, k_min=5):
        self.K = K
        self.q = q
        self.c = c
        self.samples = [[] for _ in range(K)]
        self.t = 0
        self.k_min = k_min

    def reset(self):
        self.samples = [[] for _ in range(self.K)]
        self.t = 0

    def _estimate_alpha(self, samples):

        if len(samples) < self.k_min:
            return 2.0

        x = np.sort(samples)[::-1]
        k = min(max(self.k_min, len(samples) // 10), len(samples) // 2)

        if k <= 1:
            return 2.0

        Xk1 = x[k]
        logs = np.log(x[:k] / Xk1)
        mean_log = logs.mean()

        if mean_log <= 0:
            return 2.0

        hill = 1.0 / mean_log


        N_eff = min(k, 30)
        beta = 5.0
        alpha_0 = 2.0
        alpha = (N_eff * hill + beta * alpha_0) / (N_eff + beta)

        return np.clip(alpha, 1.01, 10.0)

    def select_arm(self):
        self.t += 1
        if self.t <= self.K:
            return self.t - 1

        ucb = np.full(self.K, -np.inf)
        for i in range(self.K):
            smp = self.samples[i]
            if len(smp) == 0:
                continue


            qv = np.quantile(smp, self.q)


            alpha = self._estimate_alpha(smp)

            bonus = self.c * (np.log(self.t) / len(smp)) ** (1.0 / alpha)
            ucb[i] = qv + bonus

        return np.argmax(ucb)

    def update(self, arm, reward):
        self.samples[arm].append(reward)


class ParetoDistribution:

    def __init__(self, alpha, scale=1.0, loc=0.0):
        self.alpha = alpha
        self.scale = scale
        self.loc = loc

    def sample(self):
        return self.loc + self.scale * pareto.rvs(b=self.alpha)

    def quantile(self, q):
        return self.loc + self.scale * (1 / (1 - q)) ** (1.0 / self.alpha)

    def mean(self):
        if self.alpha <= 1:
            return float('inf')
        return self.loc + self.scale * self.alpha / (self.alpha - 1)



class CauchyDistribution:

    def __init__(self, loc, scale=1.0):
        self.loc = loc
        self.scale = scale

    def sample(self):
        return cauchy.rvs(loc=self.loc, scale=self.scale)

    def quantile(self, q):
        return cauchy.ppf(q, loc=self.loc, scale=self.scale)

    def mean(self):
        return float('inf')


class NormalDistribution:

    def __init__(self, mean, std):
        self.mean_val = mean
        self.std = std

    def sample(self):
        return np.random.normal(self.mean_val, self.std)

    def quantile(self, q):
        return norm.ppf(q, loc=self.mean_val, scale=self.std)

    def mean(self):
        return self.mean_val


class MixedDistribution:

    def __init__(self, p_heavy, alpha_heavy, mean_light, std_light):
        self.p_heavy = p_heavy
        self.pareto = ParetoDistribution(alpha=alpha_heavy)
        self.normal = NormalDistribution(mean=mean_light, std=std_light)

    def sample(self):
        if np.random.uniform() < self.p_heavy:
            return self.pareto.sample()
        return self.normal.sample()

    def quantile(self, q, n_samples=100000):
        samples = [self.sample() for _ in range(n_samples)]
        return np.quantile(samples, q)

    def mean(self):
        if self.p_heavy > 0:
            return float('inf')
        return self.normal.mean()



class BanditExperiment:
    def __init__(self, distributions, algorithms, horizon, n_simulations=30):
        self.distributions = distributions
        self.algorithms = algorithms
        self.horizon = horizon
        self.n_simulations = n_simulations
        self.n_arms = len(distributions)

    def compute_regret(self, arm, t):

        true_quantiles = [d.quantile(0.75) for d in self.distributions]
        optimal_quantile = max(true_quantiles)
        return optimal_quantile - true_quantiles[arm]

    def run_single(self, algo):

        algo.reset()
        regrets = np.zeros(self.horizon)
        cumulative = 0

        for t in range(self.horizon):
            arm = algo.select_arm()
            reward = self.distributions[arm].sample()
            algo.update(arm, reward)

            regret = self.compute_regret(arm, t)
            cumulative += regret
            regrets[t] = cumulative

        return regrets

    def run(self):

        results = {name: [] for name in self.algorithms.keys()}

        for sim in range(self.n_simulations):
            print(f"  Симуляция {sim + 1}/{self.n_simulations}")
            for name, algo in self.algorithms.items():
                regrets = self.run_single(algo)
                results[name].append(regrets)

        averaged = {}
        stds = {}
        for name, regret_list in results.items():
            averaged[name] = np.mean(regret_list, axis=0)
            stds[name] = np.std(regret_list, axis=0)

        return averaged, stds


def plot_results(results, stds, title, save_path=None):

    plt.figure(figsize=(12, 7))

    colors = {
        'FunctionalUCB': '#1f77b4',
        'ClassicUCB': '#ff7f0e',
        'QuantileUCB': '#2ca02c',
        'HeavyTailUCB': '#d62728'
    }
    linestyles = {
        'FunctionalUCB': '-',
        'ClassicUCB': '--',
        'QuantileUCB': '-.',
        'HeavyTailUCB': ':'
    }

    for name, regrets in results.items():
        color = colors.get(name, 'black')
        ls = linestyles.get(name, '-')
        plt.plot(regrets, label=name, color=color, linestyle=ls, linewidth=2)



    plt.xlabel('Шаг t', fontsize=12)
    plt.ylabel('Кумулятивное сожаление (regret)', fontsize=12)
    plt.title(title, fontsize=14)
    plt.legend(loc='upper left', fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.yscale('log')

    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


def print_distribution_info(distributions, name):

    print(f"\n{'='*60}")
    print(f"ЭКСПЕРИМЕНТ: {name}")
    print(f"{'='*60}")

    for i, d in enumerate(distributions):
        q75 = d.quantile(0.75)
        mean_val = d.mean() if hasattr(d, 'mean') else 'не определено'
        if isinstance(mean_val, float) and np.isinf(mean_val):
            mean_val = '∞'

        print(f"Рука {i + 1}:")
        print(f"  0.75-квантиль = {q75:.4f}")
        print(f"  Среднее = {mean_val}")

    optimal = np.argmax([d.quantile(0.75) for d in distributions])
    print(f"\nОптимальная рука: {optimal + 1}")



def main():
    np.random.seed(42)

    HORIZON = 3000
    N_SIMULATIONS = 250
    QUANTILE = 0.75

    algorithms = {
        'FunctionalUCB': FunctionalUCB(K=3, q=QUANTILE, c=2.0, k_min=5),
        'ClassicUCB': ClassicUCBForQuantile(K=3, q=QUANTILE, c=1.0),
        'QuantileUCB': QuantileUCB(K=3, q=QUANTILE, c=1.0),
        'HeavyTailUCB': HeavyTailUCBForQuantile(K=3, q=QUANTILE, alpha=1.5, c=1.0),
    }

    print_distribution_info([
        ParetoDistribution(alpha=1.2, scale=1.0),
        ParetoDistribution(alpha=1.8, scale=1.0),
        ParetoDistribution(alpha=2.5, scale=1.0)
    ], "Парето: α = 1.2, 1.8, 2.5")

    exp1 = BanditExperiment(
        distributions=[
            ParetoDistribution(alpha=1.2, scale=1.0),
            ParetoDistribution(alpha=1.8, scale=1.0),
            ParetoDistribution(alpha=2.5, scale=1.0)
        ],
        algorithms=algorithms,
        horizon=HORIZON,
        n_simulations=N_SIMULATIONS
    )
    results1, stds1 = exp1.run()
    plot_results(results1, stds1,
                "",
                "exp1_pareto_fixed.png")

    algorithms_heavy = {
        'FunctionalUCB': FunctionalUCB(K=3, q=QUANTILE, c=2.5, k_min=5),
        'ClassicUCB': ClassicUCBForQuantile(K=3, q=QUANTILE, c=1.0),
        'QuantileUCB': QuantileUCB(K=3, q=QUANTILE, c=1.5),
        'HeavyTailUCB': HeavyTailUCBForQuantile(K=3, q=QUANTILE, alpha=1.2, c=1.5),
    }

    print_distribution_info([
        ParetoDistribution(alpha=1.05, scale=1.0),
        ParetoDistribution(alpha=1.3, scale=1.0),
        ParetoDistribution(alpha=2.0, scale=1.0)
    ], "Парето: α = 1.05, 1.3, 2.0 (очень тяжёлые хвосты)")

    exp2 = BanditExperiment(
        distributions=[
            ParetoDistribution(alpha=1.05, scale=1.0),
            ParetoDistribution(alpha=1.3, scale=1.0),
            ParetoDistribution(alpha=2.0, scale=1.0)
        ],
        algorithms=algorithms_heavy,
        horizon=HORIZON,
        n_simulations=N_SIMULATIONS
    )
    results2, stds2 = exp2.run()
    plot_results(results2, stds2,
                "",
                "exp2_very_heavy_fixed.png")


    algorithms_cauchy = {
        'FunctionalUCB': FunctionalUCB(K=3, q=QUANTILE, c=3.0, k_min=5),
        'ClassicUCB': ClassicUCBForQuantile(K=3, q=QUANTILE, c=1.0),
        'QuantileUCB': QuantileUCB(K=3, q=QUANTILE, c=2.0),
        'HeavyTailUCB': HeavyTailUCBForQuantile(K=3, q=QUANTILE, alpha=1.0, c=2.0)
    }

    print_distribution_info([
        CauchyDistribution(loc=5.0, scale=1.0),
        CauchyDistribution(loc=4.0, scale=1.0),
        CauchyDistribution(loc=3.0, scale=1.0)
    ], "Распределение Коши (нет моментов)")

    exp3 = BanditExperiment(
        distributions=[
            CauchyDistribution(loc=5.0, scale=1.0),
            CauchyDistribution(loc=4.0, scale=1.0),
            CauchyDistribution(loc=3.0, scale=1.0)
        ],
        algorithms=algorithms_cauchy,
        horizon=HORIZON,
        n_simulations=N_SIMULATIONS
    )
    results3, stds3 = exp3.run()
    plot_results(results3, stds3,
                "",
                "exp3_cauchy_fixed.png")

    algorithms_normal = {
        'FunctionalUCB': FunctionalUCB(K=3, q=QUANTILE, c=1.0, k_min=5),
        'ClassicUCB': ClassicUCBForQuantile(K=3, q=QUANTILE, c=1.0),
        'QuantileUCB': QuantileUCB(K=3, q=QUANTILE, c=1.0),
        'HeavyTailUCB': HeavyTailUCBForQuantile(K=3, q=QUANTILE, alpha=1.0, c=2.0)

    }

    print_distribution_info([
        NormalDistribution(mean=10.0, std=1.0),
        NormalDistribution(mean=9.0, std=1.0),
        NormalDistribution(mean=8.0, std=1.0)
    ], "Нормальные распределения (лёгкие хвосты)")

    exp4 = BanditExperiment(
        distributions=[
            NormalDistribution(mean=10.0, std=1.0),
            NormalDistribution(mean=9.0, std=1.0),
            NormalDistribution(mean=8.0, std=1.0)
        ],
        algorithms=algorithms_normal,
        horizon=HORIZON,
        n_simulations=N_SIMULATIONS
    )
    results4, stds4 = exp4.run()
    plot_results(results4, stds4,
                "",
                "exp4_normal_fixed.png")


if __name__ == "__main__":
    main()